

### Goal
- find locations of forest loss with rubber as a driver

                


In [1]:
import os
from functools import partial
from timeit import default_timer as timer
from datetime import timedelta
import math
import rasterio
from rasterio.mask import mask
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape, Polygon
import pyproj
from shapely.ops import transform
import numpy as np
import pandas as pd
import geopandas as gpd
from rasterio.plot import show
import tqdm
from shapely.geometry import box
from rasterio.errors import RasterioIOError
from rasterio.enums import Resampling


In [3]:
import sys
sys.path.append('../')
from utils import get_tiles_as_gdf

In [4]:
target_years = list(range(2000, 2020))

In [5]:
forestloss_dir = '../../Vancutsem_yearloss/'

In [6]:
forestloss_tiles = get_tiles_as_gdf(forestloss_dir)

In [7]:
rubber_data_path = '/home/jan/Documents/EPFL/drivers/data/2024/rubber/Rubber_10m'

In [7]:
rubber_tiles = get_tiles_as_gdf(rubber_data_path)

### TODO

 - use similar approach as with vancutsem x mapbiomas 
 
 - for row in rubber_paths_df.iterrows():
 
     intersecting_vancutsem_tiles = ...
     for vancutsem_tile in intersecting_vancutsem_tiles:
         windowed read of the rubber tile using vancutsem_tile extent
         windowed read of the vancutsem tile using rubber tile extent
         iterate through rows similar to before 
         

In [16]:
label_file_resolution = (10,10)
target_patch_size = (1000,1000) #in meters, width, height


In [ ]:
geoms = []
forest_loss_years = []
drivers = []


for idx, row in forestloss_tiles.iterrows():
    print(idx)
    
    
    forestloss_tile_path = os.path.join(forestloss_dir, row.filename)

    with rasterio.open(forestloss_tile_path) as src:
        forestloss_profile = src.profile
        
    
    rubber_tiles_intersecting = rubber_tiles.loc[rubber_tiles.intersects(row.geometry)]
    if len(rubber_tiles_intersecting) == 0:
        continue
        
    for rubber_idx, rubber_row in rubber_tiles_intersecting.iterrows():
    
        rubber_path = os.path.join(rubber_data_path, rubber_row.filename)
        
        with rasterio.open(rubber_path) as src:
            rubber_profile = src.profile

        #Define the window of the TMF extent (to use for reading the rubber tile)
        forestloss_tile_extent_window = rasterio.windows.from_bounds(
                                              left=row.geometry.bounds[0], 
                                              bottom=row.geometry.bounds[1], 
                                              right=row.geometry.bounds[2], 
                                              top=row.geometry.bounds[3],
                                              transform=rubber_profile['transform'], 
                                             )


        #Read the rubber raster using the TMF window
        #This is so that I don't read the rubber raster outside of the TMF tile that is being used in this iteration
        with rasterio.open(rubber_path) as src:
            rubber_data = src.read(1, window=forestloss_tile_extent_window)
            rubber_bounds_orig = src.bounds

        bounds_edited = rasterio.coords.BoundingBox(
            left=max(rubber_bounds_orig.left, row.geometry.bounds[0]), 
            bottom=max(rubber_bounds_orig.bottom, row.geometry.bounds[1]), 
            right=min(rubber_bounds_orig.right, row.geometry.bounds[2]), 
            top=min(rubber_bounds_orig.top, row.geometry.bounds[3]),
        )
        
        
        #This is the correct extent of the windowed rubber data, transformed to the forestloss projection
        window = rasterio.windows.from_bounds(left=max(rubber_bounds_orig.left, row.geometry.bounds[0]), 
                                              bottom=max(rubber_bounds_orig.bottom, row.geometry.bounds[1]), 
                                              right=min(rubber_bounds_orig.right, row.geometry.bounds[2]), 
                                              top=min(rubber_bounds_orig.top, row.geometry.bounds[3]),
                                              transform=forestloss_profile['transform'], 
                                             )

        #Now read the forestloss data for the same area I read the rubber file
        #And resample to the same size

        with rasterio.open(forestloss_tile_path) as src:
            forestloss_data = src.read( 
                out_shape=(
                    src.count,
                    rubber_data.shape[0],
                    rubber_data.shape[1]
                    ),
                resampling=Resampling.bilinear,
                window=window)
            forestloss_data = forestloss_data[0]


        #As a result, rubber_data nad forestloss_data have the same extent and same pixel size
        #which allows me to compare pixel by pixel
            
        #if forestloss_data.max() == 0: #Means there is no forestloss data for the area
        #    print(f"No forestloss data for idx {idx}")
        #    continue

        
        rows, cols = rubber_data.shape
        patch_rows, patch_cols = [int(math.ceil(size/res)) for size,res in zip(target_patch_size, label_file_resolution)]


        for i in range(0, rows, patch_rows):
            for j in range(0, cols, patch_cols):
                forestloss_patch = forestloss_data[i:i+patch_rows, j:j+patch_cols]

                rubber_patch = rubber_data[i:i+patch_rows, j:j+patch_cols]

                if not np.isin(forestloss_patch, target_years).any(): #No forest loss in target period within the window
                    continue
                if rubber_patch.max() == 0: #Means there are no positive rubber data for the window
                    continue

                forest_loss_mask = (np.isin(forestloss_patch, target_years))
                
                #If I got here, it means there is forestloss and there is some rubber in the patch
                #so, i need to get the coordinates of the patch from the indices                
                
                #I do it with using the row,col indices + the transform
                #But I also need to take into account that I read the rubber tile
                #with a window -> use the forestloss_offset

                
                ##
                lower_left_coords = rasterio.transform.xy(
                    rubber_profile['transform'], 
                    i + round(max(0, forestloss_tile_extent_window.row_off)), 
                    j + round(max(0, forestloss_tile_extent_window.col_off)),  #TODO: test this works as intended
                    center='ll')
                upper_right_coords = rasterio.transform.xy(
                    rubber_profile['transform'], 
                    i + round(max(0, forestloss_tile_extent_window.row_off)) + patch_rows,
                    j + round(max(0, forestloss_tile_extent_window.col_off)) + patch_cols,  #TODO: test this works as intended
                    center='ur')

                
                _geom = box(lower_left_coords[0], lower_left_coords[1], upper_right_coords[0], upper_right_coords[1])
                assert box(*rubber_tiles_intersecting.total_bounds).intersects(_geom)
                _forest_loss_years = np.unique(forestloss_patch[forest_loss_mask], return_counts=True)
                _drivers = np.unique(rubber_patch[forest_loss_mask], return_counts=True)

                geoms.append(_geom)
                forest_loss_years.append(_forest_loss_years)
                drivers.append(_drivers)


        print(f"Items: {len(geoms)}")


0
1
2
3
4
5
6
Items: 13911
Items: 29263
7


In [32]:
gdf = gpd.GeoDataFrame(geometry=geoms)

In [33]:
gdf['forest_loss_years'] = forest_loss_years

In [13]:
pd.DataFrame(gdf).to_csv('../../data/tmp/rubber_driver.csv', index=False)

In [14]:
gdf[['geometry']].to_file('../../data/tmp/rubber_annotation###.gpkg')

### Filtering

- above I extracted all possible labeled examples
- now filter by
    - how much forest loss there was (e.g., at least 50 pixels)

In [8]:
df = pd.read_csv('../data/rubber_driver.csv')

In [9]:
import re
import ast
def modify_string_multiple_items(input_string):
    modified_string = input_string.replace('\n', '').replace(' ', '')
    # Adjusting regex to match and replace patterns for arrays with multiple items
    modified_string = re.sub(r'array\(\[([0-9, ]+)\](,*dtype=\w+\d*)?\)', r'[\1]', modified_string)
    modified_string = modified_string.replace('(','').replace('\n', '').replace(' ', '')
    # Removing any trailing unmatched parentheses
    modified_string = re.sub(r'\)+$', '', modified_string)  
    
    return ast.literal_eval(modified_string)

def convert_string_to_lists(input_string):
    try:
        lists = ast.literal_eval(input_string)
        if isinstance(lists, tuple) and len(lists) == 2 and all(isinstance(lst, list) for lst in lists):
            return lists
        else:
            return None
    except (SyntaxError, ValueError):
        return None

In [10]:
gdf = gpd.GeoDataFrame(df)

In [11]:
gdf['forest_loss_years'] = gdf['forest_loss_years'].apply(modify_string_multiple_items)

KeyboardInterrupt: 

In [ ]:
def filter_forest_loss(item, min_m2_loss=2500, resolution=label_file_resolution):
    
    min_px_loss = int(math.ceil(min_m2_loss/(label_file_resolution[0]*label_file_resolution[1])))    
    classes, counts = item
    if isinstance(classes, list):
        classes, counts = np.array(classes), np.array(counts) 
    return counts.sum() >= min_px_loss



In [ ]:
gdf_forest_loss_filtered =  gdf.loc[gdf.forest_loss_years.apply(filter_forest_loss)]


In [ ]:
gdf_forest_loss_filtered['main_driver_name'] = 'rubber'

In [75]:
gdf_forest_loss_filtered = gdf_forest_loss_filtered.drop('main_driver_name', axis=1)

In [16]:

pd.DataFrame(gdf).to_csv('../data/rubber_filtered.csv', index=False)

KeyboardInterrupt: 

## More filtering

- since it seems very noisy, i can overlay it with palm oil layer and disregard all examples that intersect with it

In [17]:
df = pd.read_csv('../data/rubber_filtered.csv')
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])
gdf = gpd.GeoDataFrame(df )


In [22]:
palm_oil_gdf = gpd.read_file('/home/jan/Documents/EPFL/drivers/data/2024/oil_plantation/GIS_files_IUCN_Global_Oil_Palm_Layer/Industrial_Oil_Palm_Globaly_Update_30Oct2017_dissolve.shp')


In [23]:
palm_oil_gdf_4326 = palm_oil_gdf.to_crs(4326)

In [24]:
palm_oil_gdf_4326_asia = palm_oil_gdf_4326.loc[palm_oil_gdf_4326.geometry.bounds.minx > 90]

In [25]:
palm_geometry = palm_oil_gdf_4326_asia.unary_union

In [26]:
rubber_outside_palm_plants = gdf.loc[~gdf.intersects(palm_geometry)]
#pd.DataFrame(rubber_outside_palm_plants).to_csv('../../data/tmp/tmp.csv', index=False)

In [27]:
pd.DataFrame(rubber_outside_palm_plants).to_csv('../data/rubber_palm_filtered.csv', index=False)

### Taking random subset to get started

In [29]:
subset = gdf.sample(1000, random_state=1)


In [31]:
pd.DataFrame(subset).to_csv('../data/labels/dataset_small/rubber_1000.csv', index=False)
#subset[['geometry', 'main_driver_name']].to_file('../../data/starting_dataset/cocoa_1000.gpkg')